# COGS 188 - Project Proposal

## Learning to Play 11×11 Hex: Self-Play Neural Networks vs. Classical Search

# Names

- Anshul Kolli
- Brycen Martin
- Cameron Chen

# Abstract

We will build agents to play 11×11 Hex and compare how well they play under limited computing time. Our main question is whether a policy value neural network trained through self-play improves Monte Carlo Tree Search. We plan to compare a random player, alpha-beta search, MCTS with random rollouts, and neural-network-guided MCTS. We will measure win rates, Elo ratings, and time per move, with each agent playing both colors. If time allows, we will test several settings for the strongest agents.


# Background

Hex is a two-player game in which players place stones on a hexagonal grid. Red tries to connect the top and bottom edges; Blue tries to connect the left and right edges. The game cannot end in a tie. Although the first player is guaranteed to have a winning strategy, the strategy is unknown for the standard 11×11 board<a name="gale"></a>[<sup>[2]</sup>](#galenote)<a name="reisch"></a>[<sup>[3]</sup>](#reischnote). Smaller boards have been solved by computer search<a name="henderson"></a>[<sup>[4]</sup>](#hendersonnote)<a name="pawlewicz"></a>[<sup>[5]</sup>](#pawlewicznote).

Early Hex programs used search methods such as alpha-beta minimax and hand designed board evaluations. For example, Hexy used an electrical-network evaluation and virtual connections<a name="anshelevich"></a>[<sup>[6]</sup>](#anshelevichnote). Monte Carlo Tree Search instead estimates moves by simulating games. UCT is a common MCTS method that balances trying promising moves with exploring less-tested ones<a name="kocsis"></a>[<sup>[7]</sup>](#kocsisnote)<a name="browne"></a>[<sup>[8]</sup>](#brownenote). The program MoHex, based on MCTS, won the Computer Olympiad Hex tournament in 2009, 2010, and 2011<a name="arneson"></a>[<sup>[9]</sup>](#arnesonnote).

More recent systems use neural networks to guide search. AlphaGo Zero and AlphaZero trained networks through self-play to predict promising moves and likely outcomes<a name="silver2017"></a>[<sup>[10]</sup>](#silver2017note)<a name="silver2018"></a>[<sup>[11]</sup>](#silver2018note). Expert Iteration applied a related approach to Hex and reported success on a 9×9 board<a name="anthony"></a>[<sup>[12]</sup>](#anthonynote). Later work used a network with additional predictions, and MoHex-3HNN won the 2018 Computer Olympiad<a name="gao"></a>[<sup>[13]</sup>](#gaonote).

These results motivate our project question: **Can a self-play-trained network improve MCTS when we have a limited compute budget?** We will compare neural search with heuristic search and MCTS on the same game.


# Problem Statement

We will build agents for 11×11 Hex and compare their playing strength. Our main question is whether a policy-value network trained through self-play improves MCTS when both methods get a similar search budget.

We will compare agents using games with balanced colors and controlled openings. Our specific questions are:

1. Does policy-value MCTS beat UCT and alpha-beta search at comparable budgets?
2. How do the agents' win rates change as we give them more simulations or time per move?
3. Which settings improve performance, and can a smaller, simpler configuration perform just as well?


# Data

We will generate data by having our agents play Hex. No external dataset is required.

**Self-play training data.** Our Python/NumPy game engine will generate positions and game outcomes. We will compare its legal moves and winners with DeepMind's OpenSpiel Hex implementation<a name="lanctot"></a>[<sup>[14]</sup>](#lanctotnote). Each training example will include the board, the MCTS move-visit distribution, and the final result. The board input will use three 11×11 planes for the current player's stones, the opponent's stones, and empty cells. We will store the player to move as the current perspective, and use a 180° rotation to augment the data. Validation data will come from separate games.

We aim to generate 3,000 self-play games, or about 200,000 positions if games average 60–90 moves.

**Evaluation records.** For each match, we will save the agents and settings, colors, opening, moves, winner, and move times. Each pairing will use the same openings with colors swapped to account for first player advantage.


# Proposed Solution

We will build a small Hex engine and compare four agents. The main comparison is between UCT and PV-MCTS.

**Game engine.** We will use Python and NumPy, with a union-find structure to check connections. We will check move legality and win detection against OpenSpiel on random games.

**Random player.** Chooses a legal move at random. This provides a simple baseline.

**Alpha-beta search.** Uses depth-limited minimax with alpha-beta pruning and a time limit. A shortest-path heuristic estimates how many empty cells each player needs to connect their edges.

**UCT.** Uses MCTS with UCB1 to balance promising moves and exploration<a name="kocsis2"></a>[<sup>[7]</sup>](#kocsisnote). At the leaves, it estimates positions by completing the game with random moves.

**PV-MCTS.** A policy-value neural network guides MCTS, following ideas from AlphaZero<a name="silver2018b"></a>[<sup>[11]</sup>](#silver2018note) and Expert Iteration<a name="anthony2"></a>[<sup>[12]</sup>](#anthonynote). The network will predict a probability for each move and the likely game outcome. During self-play, MCTS visit counts and game results become training targets.

**Compute and tuning.** We will begin with modest budgets and increase them only if time allows. We will test a small number of settings for search budget, exploration constant, network size, and learning rate. We will use multiple seeds for final comparisons where practical. Because training and tuning can be expensive, the required outcome is a working comparison of UCT and PV-MCTS.

**Evaluation and reproducibility.** We will run matches with colors swapped and report win rates with confidence intervals. We will also report Elo and time per move when the number of games supports meaningful estimates. We will test the engine on hand-built positions and random games. We will save seeds, settings, checkpoints, and match records.

# Evaluation Metrics

**Win rate (primary).** For each pair of agents, we will report the fraction of games won and a 95% Wilson confidence interval<a name="wilson"></a>[<sup>[15]</sup>](#wilsonnote). Each pairing will include games with both color assignments.

**Elo rating.** We will fit Elo ratings across the match results<a name="elo"></a>[<sup>[16]</sup>](#elonote). Elo gives a single approximate ranking across agents; we will include uncertainty estimates when we have enough games.

**Compute and speed.** We will compare agents at selected simulation budgets and report average time per move. This will show how playing strength changes as search time increases. Any estimate of how many UCT simulations match PV-MCTS will be treated as approximate.

**Network diagnostics.** For PV-MCTS, we will track value prediction error and how often its top predicted move matches the move most visited by search. These are diagnostic measures, not the main measure of playing strength.


# Ethics & Privacy

We will generate game data ourselves, so the project does not use personal information. We do not plan to connect the agent to online rated games. Training uses computing time and energy, so we will keep runs within our available resources and record compute use when practical. We will report results and limitations clearly, credit prior work, and follow the license for any code we reuse (including OpenSpiel's Apache 2.0 license).


# Team Expectations 

* **Communication:** We will use a group chat and respond within a day when possible. We will meet at least weekly and share notes if someone misses a meeting.
* **Shared work:** Each major task will have an owner, and another member will review code before it is merged. We will use our GitHub repository to share work.
* **Compute:** We will coordinate long training runs and share results and checkpoints.
* **Deadlines:** We will aim to finish internal drafts two days before course deadlines. If someone gets stuck or falls behind, they will tell the group early so we can adjust.
* **Decisions and conflict:** We will try to reach agreement, then use a majority vote if needed. We will discuss problems directly and ask the TA for help if the group cannot resolve them.
* **Effort:** We will aim for balanced contributions and adjust tasks when schedules change.


# Project Timeline Proposal

Proposed Fall 2026 schedule; we will confirm these dates against the course deadlines.

| Meeting Date | Meeting Time | Completed Before Meeting | Discuss at Meeting |
|---|---|---|---|
| 11/5 | 7 PM | Read the AlphaZero and Expert Iteration papers (all) | Finalize proposal; assign sections; set up GitHub repo and Python environment |
| 11/7 | 7 PM | Write and finalize proposal sections (all); submit proposal (Anshul) | Agree on code structure and interfaces between engine, agents, and match runner |
| 11/12 | 7 PM | Hex engine with union-find and unit tests, checked against OpenSpiel (Brycen); random and alpha-beta agents (Cameron); UCT MCTS agent (Anshul) | Review engine and baseline code; run first baseline matches; plan the network |
| 11/19 | 7 PM | Policy-value network and self-play training loop (Anshul and Cameron); match runner with paired openings, Wilson intervals, and Elo fitting (Brycen) | Inspect first training run and learning curves; debug |
| 11/24 | 7 PM | Long self-play training runs, staggered so we don't overload shared compute (all) | Short check-in before Thanksgiving; decide which runs to keep |
| 12/1 | 7 PM | Hyperparameter sweeps with 3 seeds each: simulations and exploration constants (Brycen and Cameron), network size and learning rate (Anshul) | Pick final configurations using Elo and the parsimony rule; plan figures and the report's main point |
| 12/5 | 3 PM | Final round-robin tournament, simulation-efficiency and time-per-move experiments (all); draft Results (Anshul) | Review results; draft Discussion, Limitations, and Ethics |
| 12/8 | 7 PM | Draft Abstract, Background, and Conclusion (Brycen); clean up repo, README, and figures (Cameron) | Edit the full report together |
| 12/11 | Before 11:59 PM | NA | Turn in Final Project |

# Footnotes
<a name="haywardnote"></a>1.[^](#hayward): Hayward, R. B., & Toft, B. (2019). *Hex, Inside and Out: The Full Story*. CRC Press. https://doi.org/10.1201/9780429031960<br>
<a name="galenote"></a>2.[^](#gale): Gale, D. (1979). The Game of Hex and the Brouwer Fixed-Point Theorem. *The American Mathematical Monthly, 86*(10), 818–827. https://doi.org/10.1080/00029890.1979.11994922<br>
<a name="reischnote"></a>3.[^](#reisch): Reisch, S. (1981). Hex ist PSPACE-vollständig. *Acta Informatica, 15*, 167–191. https://doi.org/10.1007/BF00288964<br>
<a name="hendersonnote"></a>4.[^](#henderson): Henderson, P., Arneson, B., & Hayward, R. B. (2009). Solving 8x8 Hex. *Proceedings of the 21st International Joint Conference on Artificial Intelligence (IJCAI)*, 505–509. https://www.ijcai.org/proceedings/2009/91<br>
<a name="pawlewicznote"></a>5.[^](#pawlewicz): Pawlewicz, J., & Hayward, R. B. (2014). Scalable Parallel DFPN Search. *Computers and Games 2013*, Lecture Notes in Computer Science 8427, 138–150. https://doi.org/10.1007/978-3-319-09165-5_12<br>
<a name="anshelevichnote"></a>6.[^](#anshelevich): Anshelevich, V. V. (2002). A Hierarchical Approach to Computer Hex. *Artificial Intelligence, 134*(1–2), 101–120. https://doi.org/10.1016/S0004-3702(01)00154-0<br>
<a name="kocsisnote"></a>7.[^](#kocsis): Kocsis, L., & Szepesvári, C. (2006). Bandit Based Monte-Carlo Planning. *European Conference on Machine Learning (ECML)*, 282–293. https://doi.org/10.1007/11871842_29<br>
<a name="brownenote"></a>8.[^](#browne): Browne, C. B., et al. (2012). A Survey of Monte Carlo Tree Search Methods. *IEEE Transactions on Computational Intelligence and AI in Games, 4*(1), 1–43. https://doi.org/10.1109/TCIAIG.2012.2186810<br>
<a name="arnesonnote"></a>9.[^](#arneson): Arneson, B., Hayward, R. B., & Henderson, P. (2010). Monte Carlo Tree Search in Hex. *IEEE Transactions on Computational Intelligence and AI in Games, 2*(4), 251–258. https://doi.org/10.1109/TCIAIG.2010.2067212<br>
<a name="silver2017note"></a>10.[^](#silver2017): Silver, D., et al. (2017). Mastering the game of Go without human knowledge. *Nature, 550*, 354–359. https://doi.org/10.1038/nature24270<br>
<a name="silver2018note"></a>11.[^](#silver2018): Silver, D., et al. (2018). A general reinforcement learning algorithm that masters chess, shogi, and Go through self-play. *Science, 362*(6419), 1140–1144. https://doi.org/10.1126/science.aar6404<br>
<a name="anthonynote"></a>12.[^](#anthony): Anthony, T., Tian, Z., & Barber, D. (2017). Thinking Fast and Slow with Deep Learning and Tree Search. *Advances in Neural Information Processing Systems (NeurIPS) 30*. https://arxiv.org/abs/1705.08439<br>
<a name="gaonote"></a>13.[^](#gao): Gao, C., Müller, M., & Hayward, R. (2018). Three-Head Neural Network Architecture for Monte Carlo Tree Search. *Proceedings of the 27th International Joint Conference on Artificial Intelligence (IJCAI)*, 3762–3768. https://doi.org/10.24963/ijcai.2018/523<br>
<a name="lanctotnote"></a>14.[^](#lanctot): Lanctot, M., et al. (2019). OpenSpiel: A Framework for Reinforcement Learning in Games. https://arxiv.org/abs/1908.09453<br>
<a name="wilsonnote"></a>15.[^](#wilson): Wilson, E. B. (1927). Probable Inference, the Law of Succession, and Statistical Inference. *Journal of the American Statistical Association, 22*(158), 209–212. https://doi.org/10.1080/01621459.1927.10502953<br>
<a name="elonote"></a>16.[^](#elo): Elo, A. E. (1978). *The Rating of Chessplayers, Past and Present*. Arco Publishing, New York.
